# 06 · LiteLLM gateway and model routing

Call any provider with one function, keep working when a provider fails, see the cost of every call, and send
each request to the cheapest model that can handle it, with a simulated quality-versus-cost trade-off to prove the
router earns its keep.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min (importing LiteLLM alone takes ~10 s) · **Needs:** [02 · Retries and fallbacks](02_retries_backoff_rate_limits_circuit_breakers.ipynb), [local LLMs with Ollama](../08_llm_apps/05_local_llms_with_ollama.ipynb)

## Why this matters in interviews

- "How would you migrate providers?", "what happens when OpenAI is down?" and "how do you cut the bill in half?"
  all have the same architectural answer: a **gateway** between your code and the providers, and a **router** that
  picks the model per request.
- Model routing is usually the largest *structural* cost saving: most traffic is easy. The interviewer's follow-up
  is always "how do you know the router is not sending hard questions to the small model?"
- LiteLLM is the most common open-source answer for the gateway layer; knowing its model naming, `Router`,
  fallbacks and proxy config is practical and quick to show.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pd03` | What is model routing and how do you build one? |
| `po12` | How would you build a model router, and what decides when to escalate? |
| `in15` | What is model routing / cascading, and how much does it save? |
| `ag31` | What is a router agent, and why is it the highest-leverage pattern? |
| `po13` | When would you deliberately ship the smaller model? |
| `in28` | What is a fallback model strategy? |
| `pd13` | What is a fallback model strategy and how do you test it? |
| `pd16` | How would you migrate from one LLM provider to another? |
| `pd09` | How do you handle a model provider deprecating a model? |

## Setup

`LITELLM_LOCAL_MODEL_COST_MAP=True` makes LiteLLM use the price list bundled with the package instead of fetching
the latest one from GitHub at import time: offline-safe and reproducible (but prices are as of the installed
version).

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import os
import random
import time
from collections import Counter

os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")
t0 = time.perf_counter()
import litellm
from litellm import Router
print(f"import litellm: {time.perf_counter() - t0:.1f} s")

import matplotlib.pyplot as plt
import numpy as np
import openai
from llm_setup import OFFLINE, make_client

MSG = [{"role": "user", "content": "What is 17*23?"}]
client.chat.completions.create(model=MODEL, messages=MSG, max_tokens=5)    # warm-up
litellm.suppress_debug_info = True

## 1. One function, every provider

LiteLLM's `completion()` takes OpenAI-style `messages` and returns an OpenAI-shaped response whatever the
provider. The provider is the **prefix of the model name**: `gpt-4.1-mini` (OpenAI is the default),
`ollama/qwen2.5:3b`, `anthropic/claude-haiku-4-5`, `bedrock/...`, `vertex_ai/...`, `azure/<deployment>`,
`openai/<model>` with an `api_base` for any OpenAI-compatible server (vLLM, LM Studio, a LiteLLM proxy).

`mock_response=` short-circuits the network entirely, which is how you unit-test code built on LiteLLM. No key is
set for Anthropic here, and the call still works:

In [ ]:
print("ANTHROPIC_API_KEY set?", "ANTHROPIC_API_KEY" in os.environ)
for model in ["gpt-4.1-mini", "ollama/qwen2.5:3b", "anthropic/claude-haiku-4-5"]:
    _, provider, _, _ = litellm.get_llm_provider(model)
    r = litellm.completion(model=model, messages=MSG, mock_response=f"mocked answer from {provider}")
    print(f"{model:<28} provider={provider:<10} -> {r.choices[0].message.content!r}  "
          f"(type {type(r).__name__}, usage {r.usage.prompt_tokens}+{r.usage.completion_tokens} tokens)")
assert r.choices[0].message.content == "mocked answer from anthropic"

For a *real* call we reuse this repo's client: LiteLLM accepts a ready-made OpenAI client via `client=`, so the same
line talks to OpenAI, Ollama or the offline stand-in, whichever `llm_setup` picked. (With your own keys you would
simply write `litellm.completion(model="anthropic/claude-haiku-4-5", ...)` and set `ANTHROPIC_API_KEY`.)

In [ ]:
r = litellm.completion(model=f"openai/{MODEL}", messages=MSG, client=client, max_tokens=20)
print("answer:", r.choices[0].message.content, "| model:", r.model)

print("streamed:", end=" ")
for chunk in litellm.completion(model=f"openai/{MODEL}", client=client, stream=True, messages=[
        {"role": "user", "content": "Summarise this.\n\nGateways hide provider differences. Routers pick models."}]):
    print(chunk.choices[0].delta.content or "", end="")
print()
mock_chunks = [c.choices[0].delta.content for c in
               litellm.completion(model="gpt-4.1-mini", messages=MSG, mock_response="streams work when mocked too", stream=True)]
print("mock stream chunks:", mock_chunks)

## 2. Cost tracking

LiteLLM ships a price map for hundreds of models and computes the cost of every response. That makes "cost per
request, per feature, per tenant" a logging exercise instead of a spreadsheet exercise. The prices are LiteLLM's
bundled list prices (they change; treat them as illustrative and check the provider's page).

In [ ]:
r = litellm.completion(model="gpt-4.1-mini", messages=MSG, mock_response="The answer is 391.")
print(f"usage: {r.usage.prompt_tokens} in / {r.usage.completion_tokens} out")
print(f"completion_cost(): ${litellm.completion_cost(completion_response=r):.8f}")
print(f"attached to the response: ${r._hidden_params['response_cost']:.8f}")
in_cost, out_cost = litellm.cost_per_token(model="gpt-4.1-mini", prompt_tokens=1_000_000, completion_tokens=1_000_000)
print(f"gpt-4.1-mini per 1M tokens: input ${in_cost:.2f}, output ${out_cost:.2f}")
assert abs(litellm.completion_cost(completion_response=r) - r._hidden_params["response_cost"]) < 1e-12

In [ ]:
catalog = ["gpt-4.1-nano", "gpt-4.1-mini", "gpt-4.1", "anthropic/claude-haiku-4-5", "anthropic/claude-sonnet-4-5",
           "ollama/llama3"]
rows = []
for m in catalog:
    try:
        info = litellm.get_model_info(m)
        rows.append((m, 1e6 * (info.get("input_cost_per_token") or 0), 1e6 * (info.get("output_cost_per_token") or 0)))
    except Exception as e:                       # a name missing from this LiteLLM version's price map
        print(f"{m}: not in the bundled price map ({type(e).__name__})")
for m, i, o in rows:
    print(f"{m:<28} ${i:6.2f} in  ${o:6.2f} out  per 1M tokens")

fig, ax = plt.subplots(figsize=(9, 3.4))
y = np.arange(len(rows))
ax.barh(y - 0.2, [i for _, i, _ in rows], height=0.4, color="#74c0fc", label="input")
ax.barh(y + 0.2, [o for _, _, o in rows], height=0.4, color="#3b5bdb", label="output")
ax.set_yticks(y, [m for m, _, _ in rows]); ax.invert_yaxis()
ax.set(xlabel="USD per 1M tokens (LiteLLM's bundled list prices)", title="Price spread across models: often 10-50x")
ax.legend(); plt.show()

Output tokens cost several times input, and the spread between a "nano" and a frontier model is one to two orders
of magnitude. That spread is what a router exploits. Local models (`ollama/...`) cost 0 per token here, which is
true at the margin and false once you count the GPU and the people running it.

## 3. `Router`: load balancing, retries and fallbacks as configuration

A `Router` holds a **model list**: named groups (`model_name`) of one or more **deployments** (`litellm_params`).
Your code asks for the group; the router picks a deployment, retries, and falls back to another group on failure.
Mocked deployments let us exercise every path offline. `mock_response="litellm.RateLimitError"` makes a deployment
fail like a rate-limited provider.

In [ ]:
random.seed(0)
router = Router(
    model_list=[
        {"model_name": "chat", "litellm_params": {"model": "gpt-4.1-mini", "mock_response": "from the US key"}},
        {"model_name": "chat", "litellm_params": {"model": "gpt-4.1-mini", "mock_response": "from the EU key",
                                                  "api_key": "sk-second-key-or-region"}},
        {"model_name": "primary", "litellm_params": {"model": "gpt-4.1", "mock_response": "litellm.RateLimitError"}},
        {"model_name": "backup", "litellm_params": {"model": "anthropic/claude-haiku-4-5", "mock_response": "from the backup"}},
    ],
    fallbacks=[{"primary": ["backup"]}],        # if every attempt on "primary" fails, try "backup"
    num_retries=1,
)
served = Counter(router.completion(model="chat", messages=MSG).choices[0].message.content for _ in range(20))
print("20 calls to group 'chat' were spread over:", dict(served))
t0 = time.perf_counter()
r = router.completion(model="primary", messages=MSG)
print(f"'primary' is rate-limited -> answered by {r.model!r}: {r.choices[0].message.content!r} ({time.perf_counter() - t0:.2f} s)")
assert len(served) == 2 and r.choices[0].message.content == "from the backup"

Two more router features worth naming (check the LiteLLM docs for the exact parameters of your version):
**cooldowns** (`allowed_fails`, `cooldown_time`) take a deployment out of rotation after repeated failures, a
per-deployment circuit breaker; and **context-window fallbacks** send a request that is too long for one model to a
long-context one:

In [ ]:
ctx_router = Router(model_list=[
    {"model_name": "small", "litellm_params": {"model": "gpt-4.1-nano", "mock_response": "litellm.ContextWindowExceededError"}},
    {"model_name": "long-context", "litellm_params": {"model": "gpt-4.1", "mock_response": "handled by the 1M-token model"}},
], context_window_fallbacks=[{"small": ["long-context"]}])
r = ctx_router.completion(model="small", messages=MSG)
print(r.model, "->", r.choices[0].message.content)
print(f"gpt-4.1 max input tokens per LiteLLM's map: {litellm.get_model_info('gpt-4.1')['max_input_tokens']:,}")

And real retries through a real client: offline we inject two 429s, turn the SDK's own retries off (retry at one
layer only, see notebook 02) and let LiteLLM retry.

In [ ]:
if OFFLINE:
    OFFLINE.fail_next(2, 429)
    before = OFFLINE.request_count
r = litellm.completion(model=f"openai/{MODEL}", messages=MSG, client=make_client(max_retries=0), num_retries=2)
print("answer after retries:", r.choices[0].message.content)
if OFFLINE:
    print("HTTP requests made:", OFFLINE.request_count - before)
    assert OFFLINE.request_count - before == 3

## 4. The LiteLLM proxy: a gateway for the whole company

The same model list can run as a separate **proxy server** that speaks the OpenAI API. Every team points its
OpenAI SDK at it; keys, budgets, rate limits, fallbacks, logging and caching are configured once. Not run here
(it is a server), but this is the whole setup:

```yaml
# config.yaml  -  run: litellm --config config.yaml --port 4000   (pip install 'litellm[proxy]')
model_list:
  - model_name: chat                      # what applications ask for
    litellm_params:
      model: openai/gpt-4.1-mini
      api_key: os.environ/OPENAI_API_KEY
      rpm: 3000                           # used for load balancing
  - model_name: chat                      # a second deployment of the same group
    litellm_params:
      model: azure/gpt-4.1-mini-eu
      api_base: os.environ/AZURE_API_BASE
      api_key: os.environ/AZURE_API_KEY
  - model_name: chat-backup
    litellm_params:
      model: anthropic/claude-haiku-4-5
      api_key: os.environ/ANTHROPIC_API_KEY
  - model_name: local
    litellm_params:
      model: ollama/qwen2.5:3b
      api_base: http://localhost:11434

router_settings:
  routing_strategy: simple-shuffle        # or least-busy, latency-based-routing, usage-based-routing
  num_retries: 2
  timeout: 30
  fallbacks: [{"chat": ["chat-backup"]}]
  allowed_fails: 3                        # then cool the deployment down
  cooldown_time: 30

litellm_settings:
  success_callback: ["langfuse"]          # send every call (with cost) to your observability tool

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY   # virtual keys with budgets per team are issued from this
```

```python
from openai import OpenAI
proxy = OpenAI(base_url="http://localhost:4000", api_key="sk-team-virtual-key")
proxy.chat.completions.create(model="chat", messages=[{"role": "user", "content": "hi"}])
```

Applications now depend on a *group name* ("chat"), not a vendor model id, which is what makes provider migrations
and model deprecations (`pd16`, `pd09`) a config change plus an eval run instead of a code change.

## 5. The gateway landscape

| Gateway | Shape | Strengths | Watch out for |
|---|---|---|---|
| **LiteLLM** (SDK + proxy) | open source, self-hosted | 100+ providers behind the OpenAI API, router, budgets, virtual keys, cost tracking | you operate it; it is on the critical path |
| **OpenRouter** | hosted marketplace | one key for many providers and open models, pay-as-you-go | data leaves via a third party; margins on price |
| **Portkey** | hosted or self-hosted gateway | routing, fallbacks, caching, guardrails, observability | another vendor in the path |
| **Kong AI Gateway** | plugins on the Kong API gateway | fits companies already on Kong: auth, rate limits, policies | LLM features newer than its core |
| **Cloud gateways** (Azure API Management, AWS Bedrock, GCP Vertex AI, Cloudflare AI Gateway) | managed, inside your cloud | IAM, networking and billing you already have | tied to one cloud's model catalogue and features |
| **Helicone** | proxy-style observability gateway | logging, caching, per-user limits by changing the base URL | must fail open; it sits in the path |

Feature sets move quickly; compare on: data residency, latency overhead, failure behaviour (fail open or closed),
and whether it speaks the OpenAI API so you can leave.

## 6. Model routing by difficulty

**In plain English:** most questions are easy. A cheap classifier looks at each request and sends the easy ones to
a small, cheap model and only the hard ones to the big one. The saving is structural: move 70% of traffic to a model
20x cheaper and spend falls by roughly two thirds.

We build the classifier from a small labelled set of queries (TF-IDF + logistic regression, trained in
milliseconds), then **simulate** quality and cost: each model answers correctly with a probability that depends on
difficulty. The numbers are assumptions stated in code, not measurements, but the method is exactly how you would
evaluate a real router on your own eval set.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

rng = random.Random(0)
things = ["refunds", "shipping", "passwords", "invoices", "the API", "subscriptions", "our SLA", "data exports"]
easy_t = ["What are your opening hours?", "How do I reset my password?", "Where can I find my invoice?",
          "Do you ship to {c}?", "What is the price of the basic plan?", "How do I contact support about {x}?",
          "Is there a student discount?", "Translate 'thank you' into {l}.", "What does {x} mean?",
          "Convert {n} km to miles.", "What is the capital of {c}?", "When does my trial end?"]
hard_t = ["Compare the trade-offs of {x} and {y} for a team of {n} engineers and recommend one.",
          "Write a Python function that parses our {x} logs and computes p95 latency per endpoint.",
          "Our {x} error rate doubled after the last deploy; walk me through how you would debug it.",
          "Draft a migration plan from {x} to {y} with risks, rollback steps and a timeline.",
          "Explain why {x} fails under high concurrency and propose two designs that fix it.",
          "Summarise the attached {n}-page contract and flag clauses that conflict with our policy on {x}.",
          "Given these monthly numbers, forecast next quarter's {x} costs and justify the assumptions.",
          "Design a database schema for {x} with multi-tenant isolation and explain the indexes."]
fill = lambda t: t.format(c=rng.choice(["Canada", "Japan", "Brazil", "Kenya"]), x=rng.choice(things),
                          y=rng.choice(things), l=rng.choice(["French", "Hindi", "Spanish"]), n=rng.randint(2, 60))
train = [(fill(rng.choice(easy_t)), 0) for _ in range(120)] + [(fill(rng.choice(hard_t)), 1) for _ in range(80)]
# A held-out set written differently from the templates, including hard-but-short and easy-but-long queries:
test = [("hi, can you tell me what time you close today?", 0), ("How much is shipping to Brazil?", 0),
        ("What's the refund window?", 0), ("Please translate 'good morning' into German.", 0),
        ("I would like to know, if possible, where I can download last month's invoice as a PDF file.", 0),
        ("Can you remind me what plan I am on and when it renews?", 0), ("What is the capital of Peru?", 0),
        ("How do I change my email address?", 0), ("Is support open on weekends?", 0), ("What does SSO mean?", 0),
        ("Prove that the square root of 2 is irrational.", 1), ("Why is our p99 so much worse than p50?", 1),
        ("Refactor this 300-line class into smaller testable units and explain each change.", 1),
        ("Which of our three vendor proposals has the lowest total cost over five years, and why?", 1),
        ("Write SQL to find customers whose spend dropped 30% quarter over quarter.", 1),
        ("Plan a zero-downtime migration of our Postgres cluster to a new region.", 1),
        ("Explain the difference between our two pricing models for enterprise buyers and recommend one.", 1),
        ("Debug: the agent calls the same tool in a loop until it hits the step cap.", 1)]
X_train, y_train = [q for q, _ in train], [y for _, y in train]
X_test, y_test = [q for q, _ in test], np.array([y for _, y in test])

router_clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(C=3.0, max_iter=1000))
router_clf.fit(X_train, y_train)
p_hard = router_clf.predict_proba(X_test)[:, 1]
print(f"router accuracy on the held-out set: {accuracy_score(y_test, p_hard > 0.5):.0%} ({len(test)} queries)")
for q, y, p in sorted(zip(X_test, y_test, p_hard), key=lambda t: -t[2])[:4] + sorted(zip(X_test, y_test, p_hard), key=lambda t: t[2])[:2]:
    print(f"  P(hard)={p:.2f}  label={'hard' if y else 'easy'}  {q[:70]}")

Now the simulation. Assumptions (edit them): the small model is right 96% of the time on easy queries and 55% on
hard ones; the large model 97% and 90%; the large model costs 20x more per request. To reduce noise we evaluate on a
larger synthetic stream: 2,000 requests, 70% easy, each with the router's probability of "hard" computed from the
query text, and 5% of those scores deliberately scrambled because real traffic is messier than templates.

In [ ]:
COST = {"small": 0.0002, "large": 0.004}                                   # $ per request
P_CORRECT = {("small", 0): 0.96, ("small", 1): 0.55, ("large", 0): 0.97, ("large", 1): 0.90}
stream_rng = np.random.default_rng(0)
labels = (stream_rng.random(2_000) < 0.30).astype(int)                     # 30% hard
queries = [fill(rng.choice(hard_t if y else easy_t)) for y in labels]
p_stream = router_clf.predict_proba(queries)[:, 1]
flip = stream_rng.random(2_000) < 0.05                                       # the classifier is imperfect on real traffic:
p_stream = np.where(flip, 1 - p_stream, p_stream)                            # 5% of scores are badly wrong
u = stream_rng.random(2_000)                                                 # same luck for every policy

def evaluate(route_large):
    tier = np.where(route_large, "large", "small")
    p = np.array([P_CORRECT[(t, y)] for t, y in zip(tier, labels)])
    return (u < p).mean(), np.mean([COST[t] for t in tier])

policies = {"all small": np.zeros(2_000, bool), "all large": np.ones(2_000, bool), "oracle router": labels == 1}
for name, mask in policies.items():
    q, c = evaluate(mask)
    print(f"{name:<14} quality {q:.1%}  cost ${c * 1e3:.2f} per 1k requests")
sweep = [(t, *evaluate(p_stream > t)) for t in np.linspace(0, 1, 41)]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([c * 1e3 for _, _, c in sweep], [q for _, q, _ in sweep], "o-", ms=3, color="#3b5bdb",
        label="classifier router (threshold swept)")
for (name, mask), marker in zip(policies.items(), ["s", "D", "*"]):
    q, c = evaluate(mask)
    ax.scatter(c * 1e3, q, s=120 if marker == "*" else 60, marker=marker, zorder=3, label=name)
t_pick = min((s for s in sweep if s[1] >= evaluate(policies["all large"])[0] - 0.02), key=lambda s: s[2])
ax.annotate(f"threshold {t_pick[0]:.2f}", (t_pick[2] * 1e3, t_pick[1]), xytext=(t_pick[2] * 1e3 + 0.6, t_pick[1] - 0.05),
            arrowprops=dict(arrowstyle="->"))
ax.set(xlabel="cost per 1,000 requests ($)", ylabel="share of answers correct (simulated)",
       title="Routing trades a little quality for a lot of cost")
ax.legend(loc="lower right", fontsize=8); plt.show()
q_large, c_large = evaluate(policies["all large"])
print(f"picked threshold {t_pick[0]:.2f}: quality {t_pick[1]:.1%} (all-large {q_large:.1%}) at "
      f"{t_pick[2] / c_large:.0%} of the all-large cost; {np.mean(p_stream > t_pick[0]):.0%} of traffic goes to the large model")
assert t_pick[2] < 0.5 * c_large and t_pick[1] >= q_large - 0.02

Reading the curve: at threshold 0 everything goes to the large model (top right); at 1 everything goes to the small
one (bottom left). The router's knee gets within two points of all-large quality for well under half the cost, and the
oracle shows the headroom a better classifier would unlock. What makes this production-grade (`po12`, `in15`):

- **The escalation signal.** A classifier (as here), rules on intent or user tier, or a **cascade**: the small model
  answers first and a verifiable check (the output parses, the cited chunk exists, retrieval returned something,
  a tool call validates) decides whether to escalate. Do not trust the model's self-reported confidence; it is
  poorly calibrated.
- **Measure per route.** Report quality separately for traffic routed small and routed large, and keep a **shadow
  sample** where the large model also answers routed-small traffic, so the quality you traded is a number.
- **Deliberately ship the smaller model** (`po13`) when it clears the product's bar on *your* eval set; not for
  irreversible actions or silent-failure paths.

In LiteLLM terms, the router's output is just the group name: `router.completion(model="small" if p < t else "large", ...)`.

## Try it yourself

**1.** Your finance team caps spend at $1.50 per 1,000 requests. Which threshold gives the best simulated quality
under that budget?

In [ ]:
# Solution — try it yourself first, then run this
affordable = [s for s in sweep if s[2] * 1e3 <= 1.50]
t_best, q_best, c_best = max(affordable, key=lambda s: s[1])
print(f"threshold {t_best:.2f}: quality {q_best:.1%} at ${c_best * 1e3:.2f} per 1k requests")
assert c_best * 1e3 <= 1.50

**2.** Build a three-tier router group in LiteLLM (`fast`, `balanced`, `frontier`, all mocked) and a function that
maps `P(hard)` to a tier with two thresholds. Route the held-out queries and count tiers.

In [ ]:
# Solution — try it yourself first, then run this
tiers = Router(model_list=[
    {"model_name": "fast", "litellm_params": {"model": "gpt-4.1-nano", "mock_response": "fast tier"}},
    {"model_name": "balanced", "litellm_params": {"model": "gpt-4.1-mini", "mock_response": "balanced tier"}},
    {"model_name": "frontier", "litellm_params": {"model": "gpt-4.1", "mock_response": "frontier tier"}},
])
def pick_tier(p, low=0.35, high=0.7):
    return "fast" if p < low else "balanced" if p < high else "frontier"
used = Counter(tiers.completion(model=pick_tier(p), messages=[{"role": "user", "content": q}]).choices[0].message.content
               for q, p in zip(X_test, p_hard))
print(dict(used))

**3.** A fallback drill (`pd13`): write a test that forces the primary group to fail and asserts that the answer
came from the backup *and* is marked as degraded. Run it like CI would, with mocks only.

In [ ]:
# Solution — try it yourself first, then run this
def answer_with_router(r: Router, messages):
    resp = r.completion(model="primary", messages=messages)
    return {"text": resp.choices[0].message.content, "model": resp.model, "degraded": resp.model != "gpt-4.1"}

def test_fallback_drill():
    out = answer_with_router(router, MSG)          # 'primary' is mocked to be rate-limited
    assert out["text"] == "from the backup" and out["degraded"], out
    return out

print("fallback drill passed:", test_fallback_drill())

if OFFLINE:                                        # reset the stand-in's knobs
    OFFLINE.latency = 0.0

## Say it in an interview

- **30 seconds:** "Applications call a gateway, not a vendor: LiteLLM's `completion()` or its OpenAI-compatible
  proxy, with a model list of groups and deployments. Retries, fallbacks to another provider, cooldowns and budgets
  are config; every response carries its cost. On top sits a router: a cheap classifier, rules or a cascade picks
  the smallest model that can handle each request, and I measure quality per route with a shadow sample on the big
  model."
- **Numbers:** price spreads between model tiers are 10-50x; routing 70% of traffic to a 20x cheaper model cuts
  spend by roughly two thirds. In this simulation the router came within two points of all-large quality at under
  half its cost (the exact figures depend entirely on the assumed accuracies).
- **Migration / deprecation:** code depends on a group name, the eval suite is pointed at the new model, shadow,
  canary, switch; budget time for prompt re-tuning because prompts do not port cleanly.
- **Follow-ups:** what is the escalation signal (verifiable checks beat self-reported confidence), how do you test
  fallbacks (mocked CI tests plus a small live traffic slice), what if the gateway fails (it is on the critical
  path: run it redundantly, or fail open to a direct provider call).
- **Traps:** routing on length alone; an unmeasured router ("a silent regression that looks like a cost win");
  retries configured in the SDK *and* LiteLLM *and* the proxy; trusting a price map without checking the date.

## Next

- [07 · Guardrails, PII and prompt injection](07_guardrails_pii_and_prompt_injection.ipynb): the other job of a gateway.
- [08 · Observability](08_observability_logging_and_tracing.ipynb): log model, route, cost and latency for every call.
- [05 · Latency percentiles and cost](05_latency_percentiles_p50_p95_and_cost.ipynb) · [evaluating agents](../12_evaluation/04_agent_evaluation.ipynb) · [A/B testing for evals](../12_evaluation/05_ab_testing_and_statistics_for_evals.ipynb).
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) · [AI tooling course](../../ai_tooling/index.html) · [interview bank](../../ai_interview_prep/index.html).